<a href="https://colab.research.google.com/github/MeisaKamiliaa/Code_Skripsi/blob/main/FINAL_SENTIMENT_FIX_BANGET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
df = pd.read_csv("ALL_SAHAM_RELEVANCY_NEWS.csv")
df

,Unnamed: 0,tanggal,kategori,judul,url,konten_clean
0,0,2022-01-05,ekonomi,"Soroti Minyak Goreng Mahal, Ma'ruf Amin: Janga...",https://finance.detik.com/berita-ekonomi-bisni...,harga minyak goreng sorot publik lantar harga...
1,1,2022-01-05,ekonomi,"Lagi Terbelit Utang, Evergrande Disuruh Hancur...",https://finance.detik.com/properti/d-5885844/l...,selesai utang kembang properti china evergran...
2,3,2022-01-05,ekonomi,Sri Mulyani Kasih Tugas Khusus ke Bos Baru LPE...,https://finance.detik.com/moneter/d-5885624/sr...,lembaga biaya ekspor indonesia lpei milik ke...
3,4,2022-01-05,ekonomi,Perbankan Tak Bisa Mendadak Setop Biayai Batu ...,https://finance.detik.com/energi/d-5885790/per...,sektor uang perban arah dukung terap ekonomi h...
4,5,2022-01-05,ekonomi,Harapan Jokowi Resmikan Pasar Johar: Ekonomi R...,https://finance.detik.com/berita-ekonomi-bisni...,presiden joko widodo resmi pasar johar kota se...
...,...,...,...,...,...,...
10643,16850,2024-12-31,ekonomi,Hal tersebut disampaikan oleh Direktur Jendera...,https://finance.detik.com/energi/d-7712214/kad...,direktur jenderal ketenagalistrikan menteri en...
10644,16851,2024-12-31,ekonomi,Menteri Keuangan Sri Mulyani Indrawati menjela...,https://finance.detik.com/berita-ekonomi-bisni...,menteri uang sri mulyani indrawati barang golo...
10645,16853,2024-12-31,ekonomi,Kemendag mempunyai sembilan gudang SRG dan sat...,https://finance.detik.com/berita-ekonomi-bisni...,kemendag sembilan gudang srg gudang sistem con...
10646,16854,2024-12-31,ekonomi,"Waspada Penipuan Mengatasnamakan Bea Cukai, In...",https://finance.detik.com/berita-ekonomi-bisni...,mei zahra temu toko online instagram nama or...


In [6]:
df=df.drop(columns=["Unnamed: 0"])
df.head()

,tanggal,kategori,judul,url,konten_clean
0,2022-01-05,ekonomi,"Soroti Minyak Goreng Mahal, Ma'ruf Amin: Janga...",https://finance.detik.com/berita-ekonomi-bisni...,harga minyak goreng sorot publik lantar harga...
1,2022-01-05,ekonomi,"Lagi Terbelit Utang, Evergrande Disuruh Hancur...",https://finance.detik.com/properti/d-5885844/l...,selesai utang kembang properti china evergran...
2,2022-01-05,ekonomi,Sri Mulyani Kasih Tugas Khusus ke Bos Baru LPE...,https://finance.detik.com/moneter/d-5885624/sr...,lembaga biaya ekspor indonesia lpei milik ke...
3,2022-01-05,ekonomi,Perbankan Tak Bisa Mendadak Setop Biayai Batu ...,https://finance.detik.com/energi/d-5885790/per...,sektor uang perban arah dukung terap ekonomi h...
4,2022-01-05,ekonomi,Harapan Jokowi Resmikan Pasar Johar: Ekonomi R...,https://finance.detik.com/berita-ekonomi-bisni...,presiden joko widodo resmi pasar johar kota se...


In [7]:
df.isna().sum()

,0
tanggal,0
kategori,0
judul,0
url,0
konten_clean,0


In [9]:
!pip install -q transformers torch pandas

import pandas as pd
import re
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# load model
MODEL_ID = "taufiqdp/indonesian-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)

print("Label mapping (id2label):", model.config.id2label)

device = 0 if torch.cuda.is_available() else -1
sentiment_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    return_all_scores=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/922 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Device set to use cpu


Label mapping (id2label): {0: 'negatif', 1: 'netral', 2: 'positif'}


/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [10]:
# batch prediksi
def predict_batch(texts, batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        preds = sentiment_pipe(batch, truncation=True, max_length=512)
        results.extend(preds)
    return results

In [11]:
#keyword saham vs override
positif_keywords = [
    "menguat", "rebound", "bullish", "surplus", "naik", "melonjak",
    "penguatan", "stabil", "optimis", "pulih", "rekor", "melambung"
]
negatif_keywords = [
    "melemah", "anjlok", "bearish", "defisit", "turun", "merosot",
    "penurunan", "tertekan", "resesi", "krisis", "minus", "collapse", "pelemahan"
]

positif_pattern = re.compile(r"\b(" + "|".join(positif_keywords) + r")\b", re.IGNORECASE)
negatif_pattern = re.compile(r"\b(" + "|".join(negatif_keywords) + r")\b", re.IGNORECASE)

def apply_keyword_override(text, model_label, model_score):
    pos_match = bool(positif_pattern.search(text))
    neg_match = bool(negatif_pattern.search(text))

    if pos_match and not neg_match:
        return "positif", 1.0
    elif neg_match and not pos_match:
        return "negatif", 1.0
    elif pos_match and neg_match:
        # kalau kedua jenis keyword ada, pilih berdasarkan jumlah kemunculan
        pos_count = len(positif_pattern.findall(text))
        neg_count = len(negatif_pattern.findall(text))
        if pos_count > neg_count:
            return "positif", 1.0
        elif neg_count > pos_count:
            return "negatif", 1.0
        else:
            # seimbang → fallback ke hasil model
            return model_label, model_score
    else:
        # tidak ada keyword → pake hasil model
        return model_label, model_score

In [12]:
df["konten_clean"] = df["konten_clean"].fillna("").astype(str)

preds = predict_batch(df["konten_clean"].tolist(), batch_size=32)
df["sentiment_label_model"] = [p["label"] for p in preds]
df["sentiment_score_model"] = [float(p["score"]) for p in preds]

# Jika id2label mapping dalam bahasa Inggris, ubah ke bahasa Indonesia:
mapping = {}
for idx, lbl in model.config.id2label.items():
    lbl_low = lbl.lower()
    if "neg" in lbl_low:
        mapping[lbl] = "negatif"
    elif "neu" in lbl_low:
        mapping[lbl] = "netral"
    elif "pos" in lbl_low:
        mapping[lbl] = "positif"
    else:
        mapping[lbl] = lbl  # fallback

# Ubah label model ke versi Indonesia
df["sentiment_label_model_id"] = df["sentiment_label_model"].map(mapping).fillna(df["sentiment_label_model"])

# override keyword
adjusted = [
    apply_keyword_override(text, lbl_id, scr)
    for text, lbl_id, scr in zip(df["konten_clean"],
                                 df["sentiment_label_model_id"],
                                 df["sentiment_score_model"])
]
df["sentiment_label"], df["sentiment_score"] = zip(*adjusted)


In [13]:
df.to_csv("df_sentiment_saham_taufiqdp.csv", index=False)
print("Selesai — file ditulis: df_sentiment_saham_taufiqdp.csv")
print("Distribusi sentiment_label:", df["sentiment_label"].value_counts())

Selesai — file ditulis: df_sentiment_saham_taufiqdp.csv
Distribusi sentiment_label: sentiment_label
positif    4365
netral     4174
negatif    2109
Name: count, dtype: int64


In [14]:
import pandas as pd
df = pd.read_csv("df_sentiment_saham_taufiqdp.csv")
df

,tanggal,kategori,judul,url,konten_clean,sentiment_label_model,sentiment_score_model,sentiment_label_model_id,sentiment_label,sentiment_score
0,2022-01-05,ekonomi,"Soroti Minyak Goreng Mahal, Ma'ruf Amin: Janga...",https://finance.detik.com/berita-ekonomi-bisni...,harga minyak goreng sorot publik lantar harga...,negatif,0.956247,negatif,positif,1.000000
1,2022-01-05,ekonomi,"Lagi Terbelit Utang, Evergrande Disuruh Hancur...",https://finance.detik.com/properti/d-5885844/l...,selesai utang kembang properti china evergran...,netral,0.656194,netral,negatif,1.000000
2,2022-01-05,ekonomi,Sri Mulyani Kasih Tugas Khusus ke Bos Baru LPE...,https://finance.detik.com/moneter/d-5885624/sr...,lembaga biaya ekspor indonesia lpei milik ke...,positif,0.715069,positif,positif,0.715069
3,2022-01-05,ekonomi,Perbankan Tak Bisa Mendadak Setop Biayai Batu ...,https://finance.detik.com/energi/d-5885790/per...,sektor uang perban arah dukung terap ekonomi h...,positif,0.878706,positif,negatif,1.000000
4,2022-01-05,ekonomi,Harapan Jokowi Resmikan Pasar Johar: Ekonomi R...,https://finance.detik.com/berita-ekonomi-bisni...,presiden joko widodo resmi pasar johar kota se...,positif,0.896946,positif,positif,0.896946
...,...,...,...,...,...,...,...,...,...,...
10643,2024-12-31,ekonomi,Hal tersebut disampaikan oleh Direktur Jendera...,https://finance.detik.com/energi/d-7712214/kad...,direktur jenderal ketenagalistrikan menteri en...,netral,0.931262,netral,positif,1.000000
10644,2024-12-31,ekonomi,Menteri Keuangan Sri Mulyani Indrawati menjela...,https://finance.detik.com/berita-ekonomi-bisni...,menteri uang sri mulyani indrawati barang golo...,netral,0.965333,netral,positif,1.000000
10645,2024-12-31,ekonomi,Kemendag mempunyai sembilan gudang SRG dan sat...,https://finance.detik.com/berita-ekonomi-bisni...,kemendag sembilan gudang srg gudang sistem con...,netral,0.612721,netral,positif,1.000000
10646,2024-12-31,ekonomi,"Waspada Penipuan Mengatasnamakan Bea Cukai, In...",https://finance.detik.com/berita-ekonomi-bisni...,mei zahra temu toko online instagram nama or...,negatif,0.940484,negatif,negatif,0.940484


In [15]:
df.isna().sum()

,0
tanggal,0
kategori,0
judul,0
url,0
konten_clean,0
sentiment_label_model,0
sentiment_score_model,0
sentiment_label_model_id,0
sentiment_label,0
sentiment_score,0


In [16]:
df.sentiment_label.value_counts()

,count
sentiment_label,
positif,4365
netral,4174
negatif,2109


In [17]:
# Pilih hanya kolom yang penting
df_final = df[[
    "tanggal", "kategori", "judul", "url",
    "konten_clean", "sentiment_label", "sentiment_score"
]]

# Simpan ke CSV baru
df_final.to_csv("df_sentiment_saham_final.csv", index=False)
print("Selesai — hasil akhir disimpan ke df_sentiment_saham_final.csv")
print("Distribusi sentimen saham:")
print(df_final["sentiment_label"].value_counts())

Selesai — hasil akhir disimpan ke df_sentiment_saham_final.csv
Distribusi sentimen saham:
sentiment_label
positif    4365
netral     4174
negatif    2109
Name: count, dtype: int64


In [18]:
df_final = pd.read_csv("df_sentiment_saham_final.csv")
df_final

,tanggal,kategori,judul,url,konten_clean,sentiment_label,sentiment_score
0,2022-01-05,ekonomi,"Soroti Minyak Goreng Mahal, Ma'ruf Amin: Janga...",https://finance.detik.com/berita-ekonomi-bisni...,harga minyak goreng sorot publik lantar harga...,positif,1.000000
1,2022-01-05,ekonomi,"Lagi Terbelit Utang, Evergrande Disuruh Hancur...",https://finance.detik.com/properti/d-5885844/l...,selesai utang kembang properti china evergran...,negatif,1.000000
2,2022-01-05,ekonomi,Sri Mulyani Kasih Tugas Khusus ke Bos Baru LPE...,https://finance.detik.com/moneter/d-5885624/sr...,lembaga biaya ekspor indonesia lpei milik ke...,positif,0.715069
3,2022-01-05,ekonomi,Perbankan Tak Bisa Mendadak Setop Biayai Batu ...,https://finance.detik.com/energi/d-5885790/per...,sektor uang perban arah dukung terap ekonomi h...,negatif,1.000000
4,2022-01-05,ekonomi,Harapan Jokowi Resmikan Pasar Johar: Ekonomi R...,https://finance.detik.com/berita-ekonomi-bisni...,presiden joko widodo resmi pasar johar kota se...,positif,0.896946
...,...,...,...,...,...,...,...
10643,2024-12-31,ekonomi,Hal tersebut disampaikan oleh Direktur Jendera...,https://finance.detik.com/energi/d-7712214/kad...,direktur jenderal ketenagalistrikan menteri en...,positif,1.000000
10644,2024-12-31,ekonomi,Menteri Keuangan Sri Mulyani Indrawati menjela...,https://finance.detik.com/berita-ekonomi-bisni...,menteri uang sri mulyani indrawati barang golo...,positif,1.000000
10645,2024-12-31,ekonomi,Kemendag mempunyai sembilan gudang SRG dan sat...,https://finance.detik.com/berita-ekonomi-bisni...,kemendag sembilan gudang srg gudang sistem con...,positif,1.000000
10646,2024-12-31,ekonomi,"Waspada Penipuan Mengatasnamakan Bea Cukai, In...",https://finance.detik.com/berita-ekonomi-bisni...,mei zahra temu toko online instagram nama or...,negatif,0.940484


In [19]:
df.sentiment_label.value_counts()

,count
sentiment_label,
positif,4365
netral,4174
negatif,2109


In [20]:
from sklearn.utils import resample

df_pos = df_final[df_final["sentiment_label"] == "positif"]
df_net = df_final[df_final["sentiment_label"] == "netral"]
df_neg = df_final[df_final["sentiment_label"] == "negatif"]

TARGET = 4000

# resampling
df_pos_bal = resample(df_pos, replace=False, n_samples=TARGET, random_state=42)  # undersample
df_net_bal = resample(df_net, replace=False, n_samples=TARGET, random_state=42)  # undersample
df_neg_bal = resample(df_neg, replace=True,  n_samples=TARGET, random_state=42)  # oversample

# gabung
df_balanced = pd.concat([df_pos_bal, df_net_bal, df_neg_bal])

# shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

df_balanced.to_csv("df_sentiment_saham_balanced.csv", index=False)
print("Selesai — data balanced disimpan ke df_sentiment_saham_balanced.csv")
print("Distribusi baru:")
print(df_balanced["sentiment_label"].value_counts())

Selesai — data balanced disimpan ke df_sentiment_saham_balanced.csv
Distribusi baru:
sentiment_label
positif    4000
netral     4000
negatif    4000
Name: count, dtype: int64


In [21]:
df=pd.read_csv("df_sentiment_saham_balanced.csv")
df

,tanggal,kategori,judul,url,konten_clean,sentiment_label,sentiment_score
0,2024-05-25,ekonomi,"Ma, Beli Wajan di Transmart Full Day Sale Aja,...",https://finance.detik.com/berita-ekonomi-bisni...,pesta diskon transmart full day sale gelar bes...,positif,0.974428
1,2024-12-08,ekonomi,"Di Transmart Full Day Sale, aneka sepeda listr...",https://finance.detik.com/berita-ekonomi-bisni...,transmart full day sale aneka sepeda listrik ...,netral,0.600188
2,2023-08-15,ekonomi,Gelaran Transmart Full Day Sale menjadi salah ...,https://finance.detik.com/berita-ekonomi-bisni...,gelar transmart full day sale salah momen nant...,positif,0.806215
3,2022-12-31,ekonomi,Pakar: Keputusan Bisnis Direksi BUMN dan Anak ...,https://finance.detik.com/detiktv/d-6491624/pa...,solution advocacy institute selenggara giat we...,negatif,0.628222
4,2024-05-05,ekonomi,Banjir Diskon Transmart Full Day Sale! Aneka K...,https://finance.detik.com/berita-ekonomi-bisni...,transmart full day sale hadir promo tarik butu...,positif,0.848727
...,...,...,...,...,...,...,...
11995,2023-10-31,ekonomi,Jalan Panjang Pertamina Luncurkan Avtur Ramah ...,https://finance.detik.com/energi/d-7012508/jal...,isu transisi energi gema iring target nol emis...,negatif,1.000000
11996,2023-02-17,ekonomi,Stafsus Erick Thohir Ungkap Alasan Waskita Tun...,https://finance.detik.com/bursa-dan-valas/d-65...,staf khusus menteri bumn arya sinulingga buka ...,netral,0.977572
11997,2024-12-11,ekonomi,KPPU-Kementerian UMKM Bahas Pengawasan Kemitra...,https://finance.detik.com/berita-ekonomi-bisni...,temu hasil urai strategi optimal mitra umkm s...,netral,0.933186
11998,2023-09-27,ekonomi,BSI Duduki Peringkat Ke-3 ESG Rating Global Is...,https://finance.detik.com/moneter/d-6954102/bs...,pt bank syariah indonesia bsi konsisten impl...,positif,0.571682


In [ ]:
df.isna().sum()

,0
tanggal,0
kategori,0
judul,0
url,0
konten_clean,0
sentiment_label,0
sentiment_score,0
